In [1]:
#the import packages
import requests
import pandas as pd
from pandas import json_normalize
import requests
import os
from pathlib import Path
from datetime import datetime, timezone,timedelta,time
from scipy import stats
import json
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
import sklearn as sklearn

In [2]:
pd.set_option("display.max_columns", None)

In [3]:
def loadDataFromFile(file_name):
    script_dir = Path().resolve().parent

    data_folder = script_dir / 'dataAnalysis and machine learning'/'data'
    print(data_folder)
    data_folder.mkdir(exist_ok=True)
    
    file_path = data_folder / (file_name + ".json")
    
    if file_path.exists():
        df = pd.read_json(file_path)
        print(f"Loaded {len(df)} records from {file_path}")
        return df
    else:
        print(f"File {file_path} does not exist.")
        return None    

In [4]:
userInputData = loadDataFromFile("UserPrevious experiments-preprocessed")
timeSeriesData_BIG = loadDataFromFile("Data:Previous experiments-preprocessed")

/home/bob/Documents/Diploma project code/Diploma-Project/dataAnalysis and machine learning/data
Loaded 79 records from /home/bob/Documents/Diploma project code/Diploma-Project/dataAnalysis and machine learning/data/UserPrevious experiments-preprocessed.json
/home/bob/Documents/Diploma project code/Diploma-Project/dataAnalysis and machine learning/data
Loaded 193493 records from /home/bob/Documents/Diploma project code/Diploma-Project/dataAnalysis and machine learning/data/Data:Previous experiments-preprocessed.json


In [5]:
# Convert back to timedelta
timeSeriesData_BIG['timestamp'] = pd.to_timedelta(timeSeriesData_BIG['timestamp'], unit='ms')

# Convert back to datetime

timeSeriesData_BIG ["datetime_timestamp"]= timeSeriesData_BIG['datetime_timestamp'].transform(
    lambda x: pd.to_datetime(x, unit='ms')
)


In [6]:
columns_datetime= [
       'date of experiment', 'actual timestamp StartingExperiment',
       'actual timestamp EndingExperiment', ]
columns_timedelta = ['time taken total',
       'timestamp InsertingSource timedelta',
       'time taken after insertion']

userInputData.loc[:,columns_datetime] = userInputData.loc[:,columns_datetime].apply(lambda x:pd.to_datetime(x, unit='ms'))
userInputData.loc[:,columns_timedelta] = userInputData.loc[:,columns_timedelta].apply(lambda x:pd.to_timedelta(x, unit='ms'))

/tmp/ipykernel_15079/207649817.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '<DatetimeArray>
['2025-07-01 00:00:00', '2025-07-02 00:00:00', '2025-07-03 00:00:00',
 '2025-07-03 00:00:00', '2025-07-09 00:00:00', '2025-07-12 00:00:00',
 '2025-07-14 00:00:00', '2025-07-19 00:00:00', '2025-07-19 00:00:00',
 '2025-07-19 00:00:00', '2025-07-19 00:00:00', '2025-07-19 00:00:00',
 '2025-07-19 00:00:00', '2025-07-20 00:00:00', '2025-07-20 00:00:00',
 '2025-07-20 00:00:00', '2025-07-20 00:00:00', '2025-07-20 00:00:00',
 '2025-07-20 00:00:00', '2025-07-20 00:00:00', '2025-07-20 00:00:00',
 '2025-07-20 00:00:00', '2025-07-20 00:00:00', '2025-07-20 00:00:00',
 '2025-07-20 00:00:00', '2025-07-20 00:00:00', '2025-07-21 00:00:00',
 '2025-07-21 00:00:00', '2025-07-21 00:00:00', '2025-07-21 00:00:00',
 '2025-07-21 00:00:00', '2025-07-21 00:00:00', '2025-07-21 00:00:00',
 '2025-07-21 00:00:00', '2025-07-22 00:00:00', '2025-07-22 

In [7]:
# Split back into dict
dict_of_timeseries = {k: v.drop(columns="keys").reset_index(drop=True) 
             for k, v in timeSeriesData_BIG.groupby("keys")}

In [8]:
userInputData

,experimentState,are-doors-opened,are-people-inside,are-windows-opened,front-wall,item-used,notes,room,side-right-wall,back-wall,side-left-wall,are-fans-on,no-source-located,timestamp InsertingSource,timestamp StartingExperiment,timestamp EndingExperiment,date of experiment,actual timestamp StartingExperiment,actual timestamp EndingExperiment,time taken total,timestamp InsertingSource timedelta,timestamp InsertingSource seconds,time taken after insertion,x axis,y axis,position of Id=0:BME680:breathVocEquivalent-x axis,position of Id=0:BME680:breathVocEquivalent-y axis,position of Id=1:BME680:breathVocEquivalent-x axis,position of Id=1:BME680:breathVocEquivalent-y axis,position of Id=2:BME680:breathVocEquivalent-x axis,position of Id=2:BME680:breathVocEquivalent-y axis
0,InsertingSourcePollutant,on,None,None,0.9,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.9,None,None,2025-07-01 16:09:41,2025-07-01 15:27:16,2025-07-01 16:14:44,2025-07-01,2025-07-01 15:27:19,2025-07-01 16:14:42,0 days 00:47:23,0 days 00:42:23,2543.0,0 days 00:05:00,-0.95,1.1,NaN,NaN,0.85,1.05,1.55,1.05
1,InsertingSourcePollutant,on,None,None,0.9,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.9,None,None,2025-07-02 15:59:29,2025-07-02 15:42:50,2025-07-02 16:04:52,2025-07-02,2025-07-02 15:42:50,2025-07-02 16:04:50,0 days 00:22:00,0 days 00:16:40,1000.0,0 days 00:05:20,-0.95,1.1,NaN,NaN,0.85,1.05,1.55,1.05
2,InsertingSourcePollutant,on,None,None,0.4,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.9,None,None,2025-07-03 12:34:32,2025-07-03 12:30:25,2025-07-03 12:45:31,2025-07-03,2025-07-03 12:30:25,2025-07-03 12:45:31,0 days 00:15:06,0 days 00:04:08,248.0,0 days 00:10:58,-0.95,1.6,NaN,NaN,0.85,1.05,1.55,1.05
3,InsertingSourcePollutant,on,None,None,0.9,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,1.8,None,None,2025-07-03 15:53:07,2025-07-03 15:43:27,2025-07-03 15:59:11,2025-07-03,2025-07-03 15:51:40,2025-07-03 15:59:10,0 days 00:07:30,0 days 00:01:28,88.0,0 days 00:06:02,-0.05,1.1,NaN,NaN,0.85,1.05,1.55,1.05
4,InsertingSourcePollutant,on,None,None,1.4,Φαρμακευτικό αλκοόλ 95%,,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,1.3,None,None,2025-07-09 19:19:19,2025-07-09 19:01:42,2025-07-09 19:27:50,2025-07-09,2025-07-09 19:01:44,2025-07-09 19:27:49,0 days 00:26:05,0 days 00:17:36,1056.0,0 days 00:08:29,-0.55,0.6,NaN,NaN,0.85,1.05,1.55,1.05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74,InsertingSourcePollutant,on,None,None,NaN,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα-Όλοι οι αισθητήρες μαζί Μ:0.80 ,...",NaN,0.6,0.6,None,None,2025-08-26 13:09:30,2025-08-26 13:02:26,2025-08-26 13:23:48,2025-08-26,2025-08-26 13:02:27,2025-08-26 13:23:47,0 days 00:21:20,0 days 00:07:04,424.0,0 days 00:14:16,-1.25,-1.4,0.9,1.2,0.90,1.20,0.90,1.20
75,InsertingSourcePollutant,on,None,None,NaN,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα-Όλοι οι αισθητήρες μαζί Μ:0.80 ,...",1.0,0.8,NaN,None,None,2025-08-26 13:42:02,2025-08-26 13:36:44,2025-08-26 13:53:04,2025-08-26,2025-08-26 13:36:46,2025-08-26 13:53:04,0 days 00:16:18,0 days 00:05:17,317.0,0 days 00:11:01,0.85,-1.2,0.9,1.2,0.90,1.20,0.90,1.20
76,InsertingSourcePollutant,on,None,None,NaN,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα-Όλοι οι αισθητήρες μαζί Μ:0.80 ,...",NaN,1.8,1.0,None,None,2025-08-26 16:49:11,2025-08-26 16:29:39,2025-08-26 17:11:16,2025-08-26,2025-08-26 16:29:41,2025-08-26 17:11:14,0 days 00:41:33,0 days 00:19:31,1171.0,0 days 00:22:02,-0.85,-0.2,0.9,1.2,0.90,1.20,0.90,1.20
77,InsertingSourcePollutant,on,None,None,0.7,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα-Όλοι οι αισθητήρες μαζί Μ:0.80 ,...",0.9,NaN,NaN,None,None,2025-08-26 18:25:06,2025-08-26 17:56:27,2025-08-26 18:33:20,2025-08-26,2025-08-26 17:56:28,2025-08-26 18:33:18,0 days 00:36:50,0 days 00:28:39,1719.0,0 days 00:08:11,0.95,1.3,0.9,1.2,0.90,1.20,

In [9]:
userInputData["room"].unique()

array(['Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55',
       'Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1',
       'Κρεβατοκάμαρα-Όλοι οι αισθητήρες μαζί Μ:0.80 , Α:0.90'],
      dtype=object)

In [10]:
userInputData_onlyRoomTogether = userInputData.loc[(userInputData["room"] == 'Κρεβατοκάμαρα-Όλοι οι αισθητήρες μαζί Μ:0.80 , Α:0.90') & (userInputData['experimentState']=='InsertingSourcePollutant'),:]
userInputData_onlyRoomTogether = userInputData_onlyRoomTogether.reset_index(names = "keys")
userInputData_onlyRoomTogether

,keys,experimentState,are-doors-opened,are-people-inside,are-windows-opened,front-wall,item-used,notes,room,side-right-wall,back-wall,side-left-wall,are-fans-on,no-source-located,timestamp InsertingSource,timestamp StartingExperiment,timestamp EndingExperiment,date of experiment,actual timestamp StartingExperiment,actual timestamp EndingExperiment,time taken total,timestamp InsertingSource timedelta,timestamp InsertingSource seconds,time taken after insertion,x axis,y axis,position of Id=0:BME680:breathVocEquivalent-x axis,position of Id=0:BME680:breathVocEquivalent-y axis,position of Id=1:BME680:breathVocEquivalent-x axis,position of Id=1:BME680:breathVocEquivalent-y axis,position of Id=2:BME680:breathVocEquivalent-x axis,position of Id=2:BME680:breathVocEquivalent-y axis
0,65,InsertingSourcePollutant,on,None,None,0.8,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα-Όλοι οι αισθητήρες μαζί Μ:0.80 ,...",NaN,NaN,1.4,None,None,2025-08-24 13:33:13,2025-08-24 13:23:53,2025-08-24 13:35:04,2025-08-24,2025-08-24 13:23:55,2025-08-24 13:35:02,0 days 00:11:07,0 days 00:09:19,559.0,0 days 00:01:48,-0.45,1.2,0.9,1.2,0.9,1.2,0.9,1.2
1,66,InsertingSourcePollutant,on,None,None,0.8,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα-Όλοι οι αισθητήρες μαζί Μ:0.80 ,...",NaN,NaN,1.4,None,None,2025-08-24 13:37:49,2025-08-24 13:35:06,2025-08-24 14:03:11,2025-08-24,2025-08-24 13:35:08,2025-08-24 14:03:09,0 days 00:28:01,0 days 00:02:42,162.0,0 days 00:25:19,-0.45,1.2,0.9,1.2,0.9,1.2,0.9,1.2
2,67,InsertingSourcePollutant,on,None,None,0.8,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα-Όλοι οι αισθητήρες μαζί Μ:0.80 ,...",NaN,NaN,1.4,None,None,2025-08-24 14:23:02,2025-08-24 14:18:44,2025-08-24 14:34:08,2025-08-24,2025-08-24 14:18:46,2025-08-24 14:34:06,0 days 00:15:20,0 days 00:04:17,257.0,0 days 00:11:03,-0.45,1.2,0.9,1.2,0.9,1.2,0.9,1.2
3,69,InsertingSourcePollutant,on,None,None,0.8,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα-Όλοι οι αισθητήρες μαζί Μ:0.80 ,...",NaN,NaN,1.4,None,None,2025-08-24 21:42:54,2025-08-24 21:39:21,2025-08-24 21:58:17,2025-08-24,2025-08-24 21:39:22,2025-08-24 21:58:15,0 days 00:18:53,0 days 00:03:33,213.0,0 days 00:15:20,-0.45,1.2,0.9,1.2,0.9,1.2,0.9,1.2
4,70,InsertingSourcePollutant,on,None,None,0.8,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα-Όλοι οι αισθητήρες μαζί Μ:0.80 ,...",NaN,NaN,1.4,None,None,2025-08-24 22:05:49,2025-08-24 22:02:36,2025-08-24 22:19:05,2025-08-24,2025-08-24 22:02:37,2025-08-24 22:19:03,0 days 00:16:26,0 days 00:03:13,193.0,0 days 00:13:13,-0.45,1.2,0.9,1.2,0.9,1.2,0.9,1.2
5,71,InsertingSourcePollutant,on,None,None,NaN,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα-Όλοι οι αισθητήρες μαζί Μ:0.80 ,...",NaN,0.6,0.6,None,None,2025-08-24 22:33:43,2025-08-24 22:23:47,2025-08-24 22:44:05,2025-08-24,2025-08-24 22:23:49,2025-08-24 22:44:03,0 days 00:20:14,0 days 00:09:55,595.0,0 days 00:10:19,-1.25,-1.4,0.9,1.2,0.9,1.2,0.9,1.2
6,73,InsertingSourcePollutant,on,None,None,0.8,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα-Όλοι οι αισθητήρες μαζί Μ:0.80 ,...",NaN,NaN,1.4,None,None,2025-08-26 11:56:26,2025-08-26 11:53:03,2025-08-26 12:44:09,2025-08-26,2025-08-26 11:53:05,2025-08-26 12:44:08,0 days 00:51:03,0 days 00:03:22,202.0,0 days 00:47:41,-0.45,1.2,0.9,1.2,0.9,1.2,0.9,1.2
7,74,InsertingSourcePollutant,on,None,None,NaN,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα-Όλοι οι αισθητήρες μαζί Μ:0.80 ,...",NaN,0.6,0.6,None,None,2025-08-26 13:09:30,2025-08-26 13:02:26,2025-08-26 13:23:48,2025-08-26,2025-08-26 13:02:27,2025-08-26 13:23:47,0 days 00:21:20,0 days 00:07:04,424.0,0 days 00:14:16,-1.25,-1.4,0.9,1.2,0.9,1.2,0.9,1.2
8,75,InsertingSourcePollutant,on,None,None,NaN,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα-Όλοι οι αισθητήρες μαζί Μ:0.80 ,...",1.0,0.8,NaN,None,None,2025-08-26 13:42:02,2025-08-26 13:36:44,2025-08-26 13:53:04,2025-08-26,2025-08-26 13:36:46,2025-08-26 13:53:04,0 days 00:16:18,0 days 00:05:17,317.0,0 days 00:11:01,0.85,-1.2,0.9,1.2,0.9,1.2,0.9,1.2
9,76,InsertingSourcePollutant,on,None,None,NaN,Φαρμακευτικό αλκοόλ 95%,,"Κρεβατοκάμαρα-Όλοι οι αισθητήρες μαζ

In [11]:
timeSeriesData_BIG_onlyRoomTogether = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"].isin(userInputData_onlyRoomTogether["keys"]),:]
timeSeriesData_BIG_onlyRoomTogether

,keys,timestamp,sensors,VOC,after_insertion,original_value,datetime_timestamp,seconds
130019,65,0 days 00:00:00,Id=0:BME680:breathVocEquivalent,3.557283,False,False,2025-08-24 13:23:55,0
130020,65,0 days 00:00:00,Id=1:BME680:breathVocEquivalent,1.062081,False,False,2025-08-24 13:23:55,0
130021,65,0 days 00:00:00,Id=2:BME680:breathVocEquivalent,3.841503,False,True,2025-08-24 13:23:55,0
130022,65,0 days 00:00:01,Id=0:BME680:breathVocEquivalent,3.544600,False,True,2025-08-24 13:23:56,1
130023,65,0 days 00:00:01,Id=1:BME680:breathVocEquivalent,1.061832,False,False,2025-08-24 13:23:56,1
...,...,...,...,...,...,...,...,...
193488,78,0 days 00:12:40,Id=2:BME680:breathVocEquivalent,733.255283,True,False,2025-08-26 18:55:47,760
193489,78,0 days 00:12:40,Id=0:BME680:breathVocEquivalent,26.946737,True,False,2025-08-26 18:55:47,760
193490,78,0 days 00:12:41,Id=1:BME680:breathVocEquivalent,81.105730,True,True,2025-08-26 18:55:48,761
193491,78,0 days 00:12:41,Id=2:BME680:breathVocEquivalent,513.575400,True,True,2025-08-26 18:55:48,761


In [12]:
dict_of_timeseries_onlyRoomTogether = {}

groups_keys = timeSeriesData_BIG_onlyRoomTogether.groupby("keys").groups

for key, indexes_of_the_timeSeriesData_BIG in groups_keys.items(): 
    mask = (
        (timeSeriesData_BIG_onlyRoomTogether.index.isin(indexes_of_the_timeSeriesData_BIG)) & 
        ((timeSeriesData_BIG_onlyRoomTogether["seconds"]) > (userInputData_onlyRoomTogether.loc[userInputData_onlyRoomTogether[userInputData_onlyRoomTogether["keys"]==key],"timestamp InsertingSource seconds"] - 20 ))  & 
        ((timeSeriesData_BIG_onlyRoomTogether["seconds"]) < (userInputData_onlyRoomTogether.loc[userInputData_onlyRoomTogether[userInputData_onlyRoomTogether["keys"]==key],"timestamp InsertingSource seconds"] + 280 ))  
        )
    dict_of_timeseries_onlyRoomTogether[key]  =  timeSeriesData_BIG_onlyRoomTogether.loc[mask,["seconds","sensors","VOC"]] 
timeSeriesData_BIG_onlyRoomTogether_FittingIntoNumpy

ValueError: Cannot index with multidimensional key

In [ ]:
sklearn

In [ ]:
# Split back into dict
dict_of_timeseries = {k: v.drop(columns="keys").reset_index(drop=True) 
             for k, v in timeSeriesData_BIG.groupby("keys")}

In [ ]:
dict_of_timeseries_onlyRoomTogether[key] 
for key,particular_experiment in dict_of_timeseries_onlyRoomTogether.items()
    sensors = particular_experiment["sensors"].unique()
    
    for sensor in sensors:
        
        transformer = RobustScaler().fit_transform(particular_experiment.loc[particular_experiment["sensors"] == sensor])